In [1]:
import pandas as pd 
df = pd.DataFrame({'billing_country':['UK', 'US', 'US', 'Canada', 'belgium', 'ireland' 'france']})

In [2]:
df

,billing_country
0,UK
1,US
2,US
3,Canada
4,belgium
5,irelandfrance


In [3]:
df['market'] = 'Rest of world'

In [4]:
df.loc[df['billing_country'] == 'UK', 'market'] ='UK'
df.loc[(df['billing_country'] == 'US') | (df['billing_country'] == 'Canada'), 'market'] = 'US'

In [5]:
df['value'] = 12, 36, 45, 2, 56, 2

In [6]:
df

,billing_country,market,value
0,UK,UK,12
1,US,US,36
2,US,US,45
3,Canada,US,2
4,belgium,Rest of world,56
5,irelandfrance,Rest of world,2


In [7]:
import polars as pl

In [8]:
df_pl = pl.from_dataframe(df)

In [9]:
df_pl

billing_country,market,value
str,str,i64
"""UK""","""UK""",12
"""US""","""US""",36
"""US""","""US""",45
"""Canada""","""US""",2
"""belgium""","""Rest of world""",56
"""irelandfrance""","""Rest of world""",2


In [10]:
df_pl.dtypes

[String, String, Int64]

In [14]:
import polars as pl

# Sample DataFrame
data = pl.DataFrame({
    "day": ["01-01-2024", "01-01-2024", "2024-01-02", "2024-01-03"],
    "total_sales": [100, 200, 150, 250],
    "net_quantity": [1, 2, 1, 3],
    "order_id": [1, 2, 3, 4],
    "customer_id": [1, 1, 2, 3],
    "discounts": [5, 10, 0, 0],
    "customer_type": ["First-time", "First-time", "Returning", "First-time"],
    "billing_country": ["USA", "USA", "Canada", "USA"]
}).with_columns(pl.col("day").str.strptime(pl.Date, "%d-%m-%Y"))

# Call compute_sales_performance
result = DashboardComputations.compute_sales_performance(data, data, "2024-01-01", "2024-01-03", "MTD")
print(result)


InvalidOperationError: conversion from `str` to `date` failed in column 'day' for 2 out of 4 values: ["2024-01-02", "2024-01-03"]

You might want to try:
- setting `strict=False` to set values that cannot be converted to `null`
- using `str.strptime`, `str.to_date`, or `str.to_datetime` and providing a format string

In [12]:
import polars as pl
import logging

class DashboardComputations:
    @staticmethod
    def compute_sales_performance(data, visits_data, start_day, end_day, period):
        if data is None or data.height == 0:
            return DashboardComputations.get_empty_performance_dict()

        # Log the data types of the columns
        logging.debug(f"Data types of sales performance data: {data.schema}")

        filtered_data = data.filter((pl.col('day') >= start_day) & (pl.col('day') <= end_day))
        visits_filtered = visits_data.filter((pl.col('day') >= start_day) & (pl.col('day') <= end_day))

        logging.info(f"Filtered data contains {filtered_data.height} rows between {start_day} and {end_day}")

        # Calculate the metrics
        if period == 'MTD':
            filtered_data = filtered_data.with_columns(
                (pl.col('day') - pl.col('day').dt.weekday().cast(pl.Duration)).alias('week_starting')
            )
            grouped_sales = filtered_data.groupby('week_starting').agg(pl.sum('total_sales').alias('total_sales'))
        else:
            grouped_sales = filtered_data.groupby('day').agg(pl.sum('total_sales').alias('total_sales'))

        # Extract dates and total sales correctly
        dates = grouped_sales['week_starting'].dt.strftime('%d/%m/%Y').to_list() if period == 'MTD' else grouped_sales['day'].dt.strftime('%d/%m/%Y').to_list()
        total_sales_by_date = grouped_sales['total_sales'].to_list()

        total_sales = float(filtered_data['total_sales'].sum())
        sold_qty = float(filtered_data['net_quantity'].sum())
        sessions = visits_filtered['total_sessions'].sum()
        transactions = filtered_data['order_id'].n_unique()

        conversion = round(float(transactions / sessions), 4) * 100 if sessions > 0 else 0
        aov = total_sales / transactions if transactions > 0 else 0

        avg_revenue_per_customer = total_sales / filtered_data['customer_id'].n_unique() if filtered_data['customer_id'].n_unique() > 0 else 0
        customer_counts = filtered_data['customer_id'].value_counts()
        returning_customers = customer_counts[customer_counts > 1].height
        total_unique_customers = customer_counts.height
        returning_customer_percentage = (
                                                returning_customers / total_unique_customers) * 100 if total_unique_customers > 0 else 0
        avg_discount = float(filtered_data['discounts'].mean()) if 'discounts' in filtered_data.columns else 0

        customer_type_counts = filtered_data['customer_type'].value_counts().to_dict()
        billing_country_counts = filtered_data['billing_country'].value_counts().to_dict()

        customer_type_data = {
            'labels': ["First-time", "Returning"],
            'values': [
                customer_type_counts.get("First-time", 0),
                customer_type_counts.get("Returning", 0)
            ]
        }

        billing_country_data = {
            'labels': list(billing_country_counts.keys()),
            'values': list(billing_country_counts.values())
        }

        logging.debug(f"Total sales: {total_sales}, Sessions: {sessions}, Transactions: {transactions}")

        return {
            'total_sales': total_sales,
            'sold_qty': sold_qty,
            'sessions': sessions,
            'transactions': transactions,
            'conversion': conversion,
            'aov': aov,
            'avg_revenue_per_customer': avg_revenue_per_customer,
            'returning_customer_percentage': returning_customer_percentage,
            'avg_discount': round(avg_discount, 2),
            'dates': dates,
            'total_sales_by_date': total_sales_by_date,
            'customer_types': customer_type_data,
            'billing_country_counts': billing_country_data
        }

    @staticmethod
    def compute_previous_period_performance(data, visits_data, start_day, end_day, period):
        filtered_data = data.filter((pl.col('day') >= start_day) & (pl.col('day') <= end_day))
        visits_filtered = visits_data.filter((pl.col('day') >= start_day) & (pl.col('day') <= end_day))

        logging.info(f"Filtered previous period data contains {filtered_data.height} rows between {start_day} and {end_day}")
        logging.debug(f"Data types of previous period data: {filtered_data.schema}")

        if period == 'MTD':
            filtered_data = filtered_data.with_columns(
                (pl.col('day') - pl.col('day').dt.weekday().cast(pl.Duration)).alias('week_starting')
            )
            grouped_sales = filtered_data.groupby('week_starting').agg(pl.sum('total_sales').alias('total_sales'))
        else:
            grouped_sales = filtered_data.groupby('day').agg(pl.sum('total_sales').alias('total_sales'))

        dates = grouped_sales['week_starting'].dt.strftime('%d/%m/%Y').to_list() if period == 'MTD' else grouped_sales['day'].dt.strftime('%d/%m/%Y').to_list()
        total_sales_by_date = grouped_sales['total_sales'].to_list()

        avg_revenue_per_customer = filtered_data['total_sales'].sum() / filtered_data['customer_id'].n_unique() if filtered_data['customer_id'].n_unique() > 0 else 0
        customer_counts = filtered_data['customer_id'].value_counts()
        returning_customers = customer_counts[customer_counts > 1].height
        total_unique_customers = customer_counts.height
        returning_customer_percentage = (
                                                returning_customers / total_unique_customers) * 100 if total_unique_customers > 0 else 0
        avg_discount = float(filtered_data['discounts'].mean()) if 'discounts' in filtered_data.columns else 0

        logging.debug(f"Previous period total sales: {filtered_data['total_sales'].sum()}, Sessions: {len(filtered_data)}")

        return {
            'total_sales': filtered_data['total_sales'].sum(),
            'sold_qty': filtered_data['net_quantity'].sum(),
            'sessions': visits_filtered['total_sessions'].sum(),
            'transactions': filtered_data['order_id'].n_unique(),
            'conversion': filtered_data['order_id'].n_unique() / visits_filtered['total_sessions'].sum() if visits_filtered['total_sessions'].sum() > 0 else 0,
            'aov': filtered_data['total_sales'].sum() / filtered_data['order_id'].n_unique() if filtered_data['order_id'].n_unique() > 0 else 0,
            'avg_revenue_per_customer': avg_revenue_per_customer,
            'returning_customer_percentage': returning_customer_percentage,
            'avg_discount': round(avg_discount, 2),
            'dates': dates,
            'total_sales_by_date': total_sales_by_date
        }

    @staticmethod
    def get_empty_performance_dict():
        return {
            'total_sales': 0,
            'sold_qty': 0,
            'sessions': 0,
            'transactions': 0,
            'conversion': 0,
            'aov': 0,
            'avg_revenue_per_customer': 0,
            'returning_customer_percentage': 0,
            'avg_discount': 0,
            'dates': [],
            'total_sales_by_date': [],
            'customer_types': {'labels': [], 'values': []},
            'billing_country_counts': {'labels': [], 'values': []},
        }
